In [1]:
# 00_generate_synthetic_data.ipynb
# Generate a reproducible synthetic additive-scorecard population for the
# quantization perturbation study. Each applicant has one active log-odds
# contribution per feature; the continuous score is their sum plus an intercept
# fixed to a target base rate. Downstream notebooks load this single file.
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(20251110)   # fixed seed for reproducibility
N, P, B = 20000, 20, 6                   # applicants, features, bins per feature
TARGET_BASE_RATE = 0.08

# feature strengths: a few strong (bureau-like) attributes, many moderate/weak
strength = np.concatenate([np.array([1.3, 1.1, 0.9, 0.8]), rng.uniform(0.2, 0.6, P - 4)])

# per-feature latent sharing a common creditworthiness signal, then binned
signal = rng.normal(size=(N, 1))
z = 0.6 * signal + 0.8 * rng.normal(size=(N, P))
edges = np.quantile(z, np.linspace(0, 1, B + 1), axis=0)
binidx = np.empty((N, P), dtype=int)
for j in range(P):
    binidx[:, j] = np.clip(np.digitize(z[:, j], edges[1:-1, j]), 0, B - 1)

# monotone per-bin log-odds contributions, centered on zero
bin_vals = (np.arange(B) - (B - 1) / 2) / ((B - 1) / 2)   # spans -1..1
C = np.empty((N, P))
for j in range(P):
    C[:, j] = strength[j] * 0.45 * bin_vals[binidx[:, j]]

# intercept via bisection to hit the target base rate
raw = C.sum(1)
mean_p = lambda b: (1.0 / (1.0 + np.exp(-(raw + b)))).mean()
lo, hi = -20.0, 20.0
for _ in range(100):
    mid = 0.5 * (lo + hi)
    if mean_p(mid) > TARGET_BASE_RATE: hi = mid
    else: lo = mid
b0 = 0.5 * (lo + hi)

f = raw + b0
p = 1.0 / (1.0 + np.exp(-f))
y = (rng.random(N) < p).astype(int)

out = pd.DataFrame(C, columns=[f"c{j}" for j in range(P)])
out["f"] = f; out["p"] = p; out["y"] = y
Path("../data").mkdir(parents=True, exist_ok=True)
out.to_parquet("../data/synthetic_population.parquet")

# numeric summary only (no file paths printed)
print("N, P, B:", N, P, B)
print("base rate:", round(float(y.mean()), 4))
print("intercept b0:", round(float(b0), 4))
print("continuous-score AUC:", round(float(roc_auc_score(y, f)), 4))
print("contribution range per cell:", round(float(C.min()), 3), "to", round(float(C.max()), 3))

N, P, B: 20000 20 6
base rate: 0.0807
intercept b0: -3.646
continuous-score AUC: 0.8652
contribution range per cell: -0.585 to 0.585
